In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load the dataset from the CSV file
csv_path = 'Dry_Bean_Dataset.csv'
df = pd.read_csv(csv_path)

# Drop rows with missing values to ensure clean training
df = df.dropna()

# Prepare features and target
X = df.drop(['Class'], axis=1)
y = df['Class']

# Encode target labels for models
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Dataset Shape after dropping NaNs: {df.shape}")
print(f"Number of Features: {X.shape[1]}")
print(f"Classes: {le.classes_}")
display(df.head())

Dataset Shape after dropping NaNs: (13611, 17)
Number of Features: 16
Classes: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715.0,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172.0,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690.0,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724.0,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417.0,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Scale data for distance-based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "K-Nearest Neighbor": KNeighborsClassifier(),
    "Naive Bayes (Gaussian)": GaussianNB(),
    "Random Forest": RandomForestClassifier()
}

results = []

for name, model in models.items():
    train_data = X_train_scaled if name in ["Logistic Regression", "K-Nearest Neighbor"] else X_train
    test_data = X_test_scaled if name in ["Logistic Regression", "K-Nearest Neighbor"] else X_test

    model.fit(train_data, y_train)
    y_pred = model.predict(test_data)
    y_proba = model.predict_proba(test_data)

    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)

    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "AUC Score": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Model,Accuracy,AUC Score,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9266,0.9950,0.9393,0.9369,0.9379,0.9117
1,Decision Tree,0.8950,0.9450,0.9080,0.9084,0.9081,0.8736
2,K-Nearest Neighbor,0.9232,0.9845,0.9395,0.9344,0.9367,0.9076
3,Naive Bayes (Gaussian),0.7580,0.9645,0.7630,0.7601,0.7596,0.7087
4,Random Forest,0.9258,0.9936,0.9379,0.9349,0.9363,0.9106


In [ ]:
import joblib
import os


# Save models and the scaler/encoder for the app
for name, model in models.items():
    filename = f"{name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, filename)

joblib.dump(scaler, 'scaler.joblib')
joblib.dump(le, 'label_encoder.joblib')

print("Models and supporting files saved to model/")

Models and supporting files saved to /content/model/
